# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through exploring the FAIR² Mass Spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The Croissant schema for this dataset is available from the following URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}\nIdentifier: {getattr(metadata, 'identifier', '')}")

## 2. Data Overview
Explore available record sets, field `@id`s, and column details using `mlcroissant` metadata.

In [ ]:
# List all RecordSets and their fields by @id
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets found in metadata!')
else:
    for rset in record_sets:
        rec_id = getattr(rset, '@id', None)
        rec_name = getattr(rset, 'name', '')
        print(f"RecordSet: {rec_name} (@id: {rec_id})")
        fields = getattr(rset, 'field', [])
        if hasattr(fields, '__iter__') and not isinstance(fields, str):
            for field in fields:
                print(f"  Field: {getattr(field, 'name', '')} (@id: {getattr(field, '@id', '')}, type: {getattr(field, 'dataType', '')})")
                columns = getattr(field, 'column', [])
                if hasattr(columns, '__iter__') and not isinstance(columns, str):
                    for col in columns:
                        print(f"    Column: {getattr(col, 'name', '')} (@id: {getattr(col, '@id', '')})")
                elif columns:
                    print(f"    Column: {getattr(columns, 'name', '')} (@id: {getattr(columns, '@id', '')})")
        else:
            print('  No fields listed for this record set.')

## 3. Data Extraction
Load data from chosen record sets into DataFrames for analysis. *All access is by `@id` as required by Croissant.*

In [ ]:
# For this dataset, let's first list available record set @id's
record_sets = getattr(metadata, 'recordSet', [])

# Collect list of record set @id's
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
print('Record sets found:', record_set_ids)

# For the FAIR^2 dataset there is likely just one main record set (we will use the first one)
if len(record_set_ids) == 0 or record_set_ids[0] is None:
    raise ValueError("No record sets with valid '@id' found in metadata.")

# You may inspect the IDs printed above and replace as desired
main_record_set_id = record_set_ids[0]

# Load all records for all record sets into pandas DataFrames
dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded {len(df)} records for record set '{rset_id}'")

print('Field columns for main record set:', dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll filter, normalize, and group the data. All references to fields are by their Croissant `@id` value.

**Don't forget to inspect the `@id`s printed in the overview and loaded columns above to select relevant fields for your analysis.**


In [ ]:
# Example: Select a numeric field for analysis -- update @id values below as needed from metadata/overview
# Let's attempt to auto-detect a likely field based on common mass spec property names

df = dataframes[main_record_set_id]
possible_numeric_fields = ['cr:field/coverage', 'cr:field/peptides', 'cr:field/MW', 'cr:field/pi', 'cr:field/abundance']
numeric_field_id = None
for col in df.columns:
    # Try an experimental match; fallback to first numeric field
    if any(c in col.lower() for c in ['coverage', 'peptide', 'mw', 'p.i', 'abundance']):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Just pick the first column (this is generic--you may wish to edit based on your metadata)
    numeric_field_id = df.columns[0]

print(f"Selected numeric field for filtering: '{numeric_field_id}'")

# Drop non-numerics from this column for operations
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].median()

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (median):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a textual/categorical field (i.e., protein family, etc.)
group_field = None
possible_group_fields = ['cr:field/description', 'cr:field/accession', 'cr:field/sample', 'cr:field/family']
for col in df.columns:
    if any(k in col.lower() for k in ['sample', 'condition', 'family', 'accession']):
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field}, mean of {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships. Update the field `@id`s as needed based on your earlier findings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If there is a group_field, visualize mean values group-wise
if group_field:
    plt.figure(figsize=(10, 4))
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    sns.barplot(data=top_groups, x=group_field, y=numeric_field_id)
    plt.xticks(rotation=45)
    plt.title(f"Top 10 {group_field} by mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
We have loaded and inspected the FAIR² Mass Spectrometry dataset using `mlcroissant` by referencing all schema, record set, field, and column elements by their Croissant `@id`. You can further expand the notebook to perform specialized analyses, such as cross-sample normalization, comparison of protein abundance statistics, etc.

Always refer back to the `@id` fields in metadata for consistent and reproducible analyses.
